# Machine Learning Digital Twin: Dialyzer Optimization
Here, we use our Random Forest Regressor trained on 1000 PDE simulations to optimize the process instantly.

In [ ]:
import sys
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../src'))

# Load the trained ML Model and the dataset
model_path = '../data/rf_surrogate.pkl'
with open(model_path, 'rb') as f:
    rf_model = pickle.load(f)

df = pd.read_csv('../data/dialyzer_dataset.csv')


### 1. Evaluate Surrogate Model Accuracy

In [ ]:
from sklearn.metrics import r2_score

X = df[['Qb_ml_min', 'Qd_ml_min', 'Quf_ml_min']]
y_true_urea = df['Clearance_Urea']
y_true_b12 = df['Clearance_B12']

y_pred = rf_model.predict(X)

plt.figure(figsize=(12, 5))

# Parity plot for Urea
plt.subplot(1, 2, 1)
plt.scatter(y_true_urea, y_pred[:, 0], alpha=0.5, c='blue')
plt.plot([y_true_urea.min(), y_true_urea.max()], [y_true_urea.min(), y_true_urea.max()], 'k--', lw=2)
plt.title(f"Urea Prediction (R² = {r2_score(y_true_urea, y_pred[:,0]):.3f})")
plt.xlabel("True Clearance (Physics Model)")
plt.ylabel("Predicted Clearance (ML Model)")
plt.grid(True, alpha=0.3)

# Parity plot for B12
plt.subplot(1, 2, 2)
plt.scatter(y_true_b12, y_pred[:, 1], alpha=0.5, c='green')
plt.plot([y_true_b12.min(), y_true_b12.max()], [y_true_b12.min(), y_true_b12.max()], 'k--', lw=2)
plt.title(f"VitB12 Prediction (R² = {r2_score(y_true_b12, y_pred[:,1]):.3f})")
plt.xlabel("True Clearance (Physics Model)")
plt.ylabel("Predicted Clearance (ML Model)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2. Rapid Process Optimization
Using the ML model, we can evaluate a grid of millions of operating conditions in seconds to find the best flow rates.

In [ ]:
# We want to maximize B12 Clearance, but Dialysate flow (Qd) costs money.
# Let's plot B12 Clearance vs Qd, for a fixed Qb = 300, Quf = 10.

Qds = np.linspace(200, 800, 100)
X_test = pd.DataFrame({
    'Qb_ml_min': np.full_like(Qds, 300),
    'Qd_ml_min': Qds,
    'Quf_ml_min': np.full_like(Qds, 10)
})

predictions = rf_model.predict(X_test)
clearance_b12_predicted = predictions[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(Qds, clearance_b12_predicted, 'g-', lw=3)
plt.title("Optimization: Effect of Dialysate Flow on B12 Clearance
(Qb = 300 mL/min, Quf = 10 mL/min)", fontsize=14)
plt.xlabel("Dialysate Flow Rate, Qd (mL/min)", fontsize=12)
plt.ylabel("Predicted VitB12 Clearance (mL/min)", fontsize=12)
plt.grid(True, alpha=0.3)

# Optimal point where curve flattens (diminishing returns)
plt.axvline(500, color='red', linestyle='--', label='Diminishing Returns start ~500 mL/min')
plt.legend()
plt.show()
